# Feature extraction 02b — Conversation

**Input:** aligned utterance files (`Audio_<ID>_aligned.csv`, one per team, from the alignment notebook) and the 25 Hz
merged file (only for the start and end time of each task block). Neither is in the repository.
**Output:** `data/speech_features_by_block.csv` — one row per team × reliability block, conversation features of the
Commander (`_C`) and the Intelligence Support operator (`_IS`).

Features are computed from the utterance table, not from the 25 Hz grid. Transcript rows can span several sentences
with pauses between them; the grid would count those pauses as speaking time.

**Rules**
- An utterance belongs to a block if its aligned start time lies inside the block (same block windows as the eye data).
- **Text features use all utterances** (transcripts were cleaned by a researcher).
- **Timing features use only `usable` utterances** (`usable` = the aligner's start and end times passed the checks
  against Whisper) and their speech segments (pauses longer than 0.75 s removed).
- Researcher speech is not used. Teams excluded for conversation data are listed in the settings cell.
- Sentiment: **sentimentr** (Rinker; R package), scored **per sentence** by `feature_extraction/02a_conversation_sentiment.R`, because one
  transcript row can contain many sentences. sentimentr adjusts for negation and intensifiers ("not the correct
  truck" is negative). It has not been validated against human ratings on this task (see the check at the end).
  Run `feature_extraction/02a_conversation_sentiment.R` first.

In [1]:
import os
from pathlib import Path
# Run from anywhere inside the repository: paths below are relative to the repository root
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir() and (p / "feature_extraction").is_dir())
os.chdir(REPO)

import os
import re
import glob
import numpy as np
import pandas as pd

BASE = "/Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV"
UTT_DIR = os.path.join(BASE, "Audio", "Aligned")              # Audio_<ID>_aligned.csv
RAW_PATH = os.path.join(BASE, "physio_eye_survey.csv")         # only used for block start/end times
SENT_PATH = os.path.join(BASE, "Audio", "sentence_sentiment.csv")   # from 02a_conversation_sentiment.R
# Word lexicons for exploratory features (`models/02_trust_two_stage.ipynb`, Section 6); not in the repository
AFINN_PATH = os.path.join(BASE, "lexicons", "AFINN-en-165.txt")          # Nielsen (2011), github.com/fnielsen/afinn
NRC_VAD_PATH = os.path.join(BASE, "lexicons", "NRC-VAD-Lexicon.txt")      # Mohammad (2018), saifmohammad.com
OUT_PATH = "data/speech_features_by_block.csv"

EXCLUDE_TEAMS = [1, 2, 3, 7, 8, 9, 11, 13, 47, 50, 51, 53, 59]  # conversation data not usable
NEG_THRESHOLD = -0.05                                          # sentence counted as negative below this score
RESPONSE_WINDOW_S = (-1.0, 5.0)    # a Commander utterance starting this soon after the IS operator stops = response
MIN_RESPONSES = 3                  # fewer responses in a block -> latency missing

SPEAKERS = {"commander": "C", "intelligence support": "IS", "intelligent support": "IS", "researcher": "R"}
BLOCK_LABEL = {"Reliable": "HH", "Mix": "M", "Unreliable": "LL"}   # transcript labels, used only as a check

## 1. Load utterances and block windows

In [2]:
files = sorted(glob.glob(os.path.join(UTT_DIR, "Audio_*_aligned.csv")))
utt = pd.concat([pd.read_csv(f, dtype={"start_unix_ns": "Int64", "end_unix_ns": "Int64"}, low_memory=False)
                 for f in files], ignore_index=True)
utt["row"] = utt.groupby("ID").cumcount()                      # transcript order within team
utt["speaker"] = utt["Speaker"].astype(str).str.strip().str.lower().map(SPEAKERS)
print(f"{len(files)} teams, {len(utt):,} utterances | unrecognised speaker labels:",
      utt.loc[utt["speaker"].isna(), "Speaker"].value_counts().to_dict())
utt = utt[~utt["ID"].isin(EXCLUDE_TEAMS) & utt["speaker"].isin(["C", "IS"])].copy()
utt["usable"] = utt["usable"].astype(bool)
print(f"after exclusions: {utt['ID'].nunique()} teams, {len(utt):,} Commander and IS utterances")

grid = pd.read_csv(RAW_PATH, usecols=["ID", "Reliability", "unix_ns"], engine="pyarrow")
windows = (grid[grid["Reliability"].isin(["HH", "M", "LL"])]
           .groupby(["ID", "Reliability"])["unix_ns"].agg(b0="min", b1="max").reset_index())
windows["minutes"] = (windows["b1"] - windows["b0"]) / 6e10

55 teams, 16,963 utterances | unrecognised speaker labels: {'D': 26}
after exclusions: 46 teams, 11,671 Commander and IS utterances


In [3]:
# Assign each utterance to the task block containing its start time
utt["Reliability"] = pd.NA
for _, w in windows.iterrows():
    inside = (utt["ID"] == w["ID"]) & utt["start_unix_ns"].between(w["b0"], w["b1"])
    utt.loc[inside, "Reliability"] = w["Reliability"]
task = utt[utt["Reliability"].notna()].copy()
print(f"{len(task):,} utterances inside task blocks, {task['ID'].nunique()} teams")

# Check: does the timing-based block agree with the transcriber's block label?
lab = task["Block"].astype(str).str.strip().map(BLOCK_LABEL)
agree = (lab == task["Reliability"])[lab.notna()]
print(f"agreement with transcript block labels: {agree.mean():.1%} of {agree.size:,} labelled utterances")
pd.crosstab(task["Reliability"], task["Block"].astype(str).str.strip()).loc[["HH", "M", "LL"]]

7,925 utterances inside task blocks, 44 teams
agreement with transcript block labels: 100.0% of 7,326 labelled utterances


Block,B1 Prediction,Mix,Reliable,Unreliable,nan
Reliability,,,,,
HH,2,0,2160,0,214
M,2,2522,0,0,204
LL,2,0,0,2644,175


## 2. Sentences, sentiment and UAV mentions

Sentences and their sentimentr scores come from `feature_extraction/02a_conversation_sentiment.R` (negative < 0 < positive; most sentences
fall between −1 and +1).
A sentence counts as **mentioning a UAV** by a keyword rule (North UAV = left display, South UAV = right display):
- North: *north*, or *on/to/at/from the left*, or *left side/screen/drone/UAV/camera/feed/video/window/monitor*
- South: *south*, or the same phrases with *right*

Bare *left* and *right* are not counted: *left* is often a verb ("it just left") and *right* usually means correct or
agreement. The examples printed below are for checking the rule.

In [4]:
PLACE = r"(?:side|screen|drone|uav|camera|feed|video|window|monitor)"
UAV_RULES = {"north": re.compile(rf"\bnorth\b|\b(?:on|to|at|from) the left\b|\bleft {PLACE}\b", re.I),
             "south": re.compile(rf"\bsouth\b|\b(?:on|to|at|from) the right\b|\bright {PLACE}\b", re.I)}

scores = pd.read_csv(SENT_PATH)
sent = task[["ID", "row", "Reliability"]].merge(scores, on=["ID", "row"], how="inner")
n_utt_scored = sent[["ID", "row"]].drop_duplicates().shape[0]
print(f"{len(task) - n_utt_scored} task utterances without a scored sentence (no letters in the text)")
sent["sentence"] = sent["sentence"].astype(str)
sent["words"] = sent["sentence"].str.findall(r"[A-Za-z']+").str.len()
for uav, rx in UAV_RULES.items():
    sent[uav] = sent["sentence"].str.contains(rx)

# Exploratory lexicon scores per sentence: AFINN (sum of word scores, -5..+5 per word; no negation handling) and
# NRC-VAD arousal and dominance (mean over words found in the lexicon, 0..1)
afinn = pd.read_csv(AFINN_PATH, sep="\t", header=None, names=["w", "s"], quoting=3)
afinn = dict(zip(afinn["w"], afinn["s"]))
nrc = pd.read_csv(NRC_VAD_PATH, sep="\t", header=None, names=["w", "V", "A", "D"], quoting=3).set_index("w")
nrc = nrc[["A", "D"]].to_dict("index")
tokens = sent["sentence"].str.lower().str.findall(r"[a-z']+")
sent["afinn"] = [sum(afinn.get(t, 0) for t in ws) for ws in tokens]
for k in "AD":
    sent[f"nrc_{k}"] = [np.mean([nrc[t][k] for t in ws if t in nrc]) if any(t in nrc for t in ws) else np.nan
                        for ws in tokens]
print(f"{len(sent):,} sentences | mention North: {sent['north'].mean():.1%} | mention South: {sent['south'].mean():.1%}"
      f" | both: {(sent['north'] & sent['south']).mean():.1%}")
pd.set_option("display.max_colwidth", 120)
sent.loc[sent["north"] | sent["south"], ["speaker", "north", "south", "sentiment", "sentence"]].sample(12, random_state=0)

12 task utterances without a scored sentence (no letters in the text)
17,707 sentences | mention North: 6.1% | mention South: 6.8% | both: 0.5%


,speaker,north,south,sentiment,sentence
1715,C,False,True,0.000000,Anything happening in south and it detected it.
2448,IS,True,False,0.000000,You got two more on north.
10232,IS,False,True,0.326599,Got that one on the right.
17477,IS,False,True,0.306186,Reaper south did a good job.
12820,IS,True,False,0.316228,I think north couldn't see the window on the top.
4308,IS,False,True,0.000000,"I think, on South."
2748,IS,False,True,-0.288675,"Yeah, south missing."
2807,IS,False,True,-0.603738,Another false alarm on south.
5572,C,False,True,0.000000,"Oh, did it get that for south?"
8771,C,True,False,0.000000,North.


## 3. Features per team × block

| Feature (for `_C` and `_IS`) | Definition |
|---|---|
| `words_per_min` | words spoken ÷ block minutes |
| `words_per_utt` | mean words per utterance |
| `sentiment_mean` | mean sentimentr score over sentences |
| `neg_share` | share of sentences scoring below −0.05 |
| `north_sentiment`, `south_sentiment` | mean score of sentences mentioning that UAV (missing if none) |
| `north_mentions_per_min`, `south_mentions_per_min` | sentences mentioning that UAV ÷ block minutes |
| `north_afinn`, `south_afinn` | exploratory: mean AFINN score of sentences mentioning that UAV |
| `north_nrc_A`, `south_nrc_A`, `north_nrc_D`, `south_nrc_D` | exploratory: mean NRC-VAD arousal / dominance of those sentences |

| Team feature | Definition |
|---|---|
| `speaking_share_C` | Commander's share of the team's speaking time (speech segments of usable utterances) |
| `switches_per_min` | changes of speaker between consecutive Commander/IS utterances (transcript order) ÷ block minutes |
| `response_latency_C` | median time from the end of an IS utterance to the start of the next Commander utterance, if it starts within −1 to 5 s (usable utterances, speech segments; missing if fewer than 3) |

Missing values (no mentions of a UAV, too few responses) are left missing here and imputed inside the model
pipelines in `models/02_trust_two_stage.ipynb`, using training folds only.

In [5]:
def segments(s):
    if not isinstance(s, str) or not s:
        return []
    return [tuple(int(x) for x in p.split("-")) for p in s.split(";") if p]

task["segs"] = task["speech_segments_unix_ns"].map(segments)
task["seg_s"] = task["segs"].map(lambda sg: sum(b - a for a, b in sg) / 1e9)
keys = ["ID", "Reliability"]
F = windows[windows["ID"].isin(task["ID"].unique())][keys + ["minutes"]].copy()

for s in ["C", "IS"]:
    ss = sent[sent["speaker"] == s]
    uu = task[task["speaker"] == s]
    f = (ss.groupby(keys).agg(words=("words", "sum"), sentiment_mean=("sentiment", "mean"),
                              neg_share=("sentiment", lambda x: (x < NEG_THRESHOLD).mean())))
    f["n_utt"] = uu.groupby(keys).size()
    for uav in ["north", "south"]:
        m = ss[ss[uav]]
        f[f"{uav}_sentiment"] = m.groupby(keys)["sentiment"].mean()
        f[f"{uav}_mentions"] = m.groupby(keys).size()
        for lx in ["afinn", "nrc_A", "nrc_D"]:
            f[f"{uav}_{lx}"] = m.groupby(keys)[lx].mean()
    f = f.reset_index()
    f = F[keys + ["minutes"]].merge(f, on=keys, how="left")
    for col in ["words", "n_utt", "north_mentions", "south_mentions"]:
        f[col] = f[col].fillna(0)
    out = pd.DataFrame({
        f"words_per_min_{s}": f["words"] / f["minutes"],
        f"words_per_utt_{s}": f["words"] / f["n_utt"].replace(0, np.nan),
        f"sentiment_mean_{s}": f["sentiment_mean"],
        f"neg_share_{s}": f["neg_share"],
        f"north_sentiment_{s}": f["north_sentiment"],
        f"south_sentiment_{s}": f["south_sentiment"],
        f"north_mentions_per_min_{s}": f["north_mentions"] / f["minutes"],
        f"south_mentions_per_min_{s}": f["south_mentions"] / f["minutes"],
        **{f"{uav}_{lx}_{s}": f[f"{uav}_{lx}"] for uav in ["north", "south"] for lx in ["afinn", "nrc_A", "nrc_D"]}})
    F = pd.concat([F.reset_index(drop=True), out], axis=1)

# Team timing features (usable utterances only)
use = task[task["usable"]]
spk = use.groupby(keys + ["speaker"])["seg_s"].sum().unstack()
F = F.merge((spk["C"] / spk.sum(axis=1)).rename("speaking_share_C").reset_index(), on=keys, how="left")

order = task.sort_values(["ID", "row"])
switches = order.groupby(keys)["speaker"].apply(lambda x: (x != x.shift()).iloc[1:].sum()).rename("switches")
F = F.merge(switches.reset_index(), on=keys, how="left")
F["switches_per_min"] = F["switches"].fillna(0) / F["minutes"]


def latency(block):
    b = block.sort_values("start_unix_ns")
    out = []
    for prev, nxt in zip(b.itertuples(), b.iloc[1:].itertuples()):
        if prev.speaker == "IS" and nxt.speaker == "C" and prev.segs and nxt.segs:
            lat = (nxt.segs[0][0] - prev.segs[-1][1]) / 1e9
            if RESPONSE_WINDOW_S[0] <= lat <= RESPONSE_WINDOW_S[1]:
                out.append(lat)
    return np.median(out) if len(out) >= MIN_RESPONSES else np.nan

lat = use.groupby(keys).apply(latency, include_groups=False).rename("response_latency_C")
F = F.merge(lat.reset_index(), on=keys, how="left").drop(columns=["switches"])

SPEECH_FEATURES = [c for c in F.columns if c not in keys + ["minutes"]]
print(f"{len(F)} blocks, {F['ID'].nunique()} teams, {len(SPEECH_FEATURES)} features")
F[SPEECH_FEATURES].isna().mean().round(3).rename("share missing").to_frame().T

132 blocks, 44 teams, 31 features


,words_per_min_C,words_per_utt_C,sentiment_mean_C,neg_share_C,north_sentiment_C,south_sentiment_C,north_mentions_per_min_C,south_mentions_per_min_C,north_afinn_C,north_nrc_A_C,north_nrc_D_C,south_afinn_C,south_nrc_A_C,south_nrc_D_C,words_per_min_IS,words_per_utt_IS,sentiment_mean_IS,neg_share_IS,north_sentiment_IS,south_sentiment_IS,north_mentions_per_min_IS,south_mentions_per_min_IS,north_afinn_IS,north_nrc_A_IS,north_nrc_D_IS,south_afinn_IS,south_nrc_A_IS,south_nrc_D_IS,speaking_share_C,switches_per_min,response_latency_C
share missing,0.0,0.038,0.038,0.038,0.348,0.424,0.0,0.0,0.348,0.348,0.348,0.424,0.424,0.424,0.0,0.023,0.023,0.023,0.341,0.341,0.0,0.0,0.341,0.341,0.341,0.341,0.341,0.341,0.053,0.0,0.28


In [6]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
F.drop(columns="minutes").to_csv(OUT_PATH, index=False)
print("saved", OUT_PATH, F.shape)

# Sanity check: feature means by condition
F.groupby("Reliability")[SPEECH_FEATURES].mean().loc[["HH", "M", "LL"]].T.round(3)

saved data/speech_features_by_block.csv (132, 34)


Reliability,HH,M,LL
words_per_min_C,20.788,24.138,25.875
words_per_utt_C,8.382,8.774,8.706
sentiment_mean_C,0.048,0.035,-0.010
neg_share_C,0.154,0.196,0.238
north_sentiment_C,0.027,0.031,-0.014
south_sentiment_C,0.169,0.029,-0.045
north_mentions_per_min_C,0.201,0.254,0.314
south_mentions_per_min_C,0.137,0.174,0.264
north_afinn_C,0.208,0.275,0.079
north_nrc_A_C,0.355,0.353,0.356


## 4. Check: how sentimentr scores task phrases

"False positive(s)/negative(s)" are scored as negative phrases (set in `feature_extraction/02a_conversation_sentiment.R`). Remaining weak
spots: descriptions without evaluative words ("it's not seeing those cars") score 0, and *right* is positive even when
it only means agreement.

In [7]:
checks = {"false positive / negative": r"\bfalse (?:positive|negative)s?\b",
          "correct / incorrect": r"\b(?:in)?correct(?:ly)?\b",
          "missed / missing": r"\bmiss(?:ed|ing|es)?\b",
          "wrong": r"\bwrong\b"}
rows = []
for name, rx in checks.items():
    hit = sent["sentence"].str.contains(rx, case=False)
    rows.append({"phrase": name, "sentences": int(hit.sum()), "share": hit.mean(),
                 "mean sentiment": sent.loc[hit, "sentiment"].mean(),
                 "share scored 0": (sent.loc[hit, "sentiment"] == 0).mean()})
pd.DataFrame(rows).round(3)

,phrase,sentences,share,mean sentiment,share scored 0
0,false positive / negative,142,0.008,-0.342,0.000
1,correct / incorrect,418,0.024,0.247,0.000
2,missed / missing,1160,0.066,-0.262,0.025
3,wrong,286,0.016,-0.365,0.000
